In [1]:

!pip install nba_api -q

import pandas as pd
import numpy as np
from nba_api.stats.endpoints import playercareerstats

# 1. Define Historical Salary Caps (Approximated for the last 15 years)
# In a production environment, you'd scrape this, but for our model, constants are faster.
salary_cap_history = {
    2010: 58044000, 2011: 58044000, 2012: 58044000, 2013: 58679000,
    2014: 63065000, 2015: 70000000, 2016: 94143000, 2017: 99093000,
    2018: 101869000, 2019: 109140000, 2020: 109140000, 2021: 112414000,
    2022: 123655000, 2023: 136021000, 2024: 140588000, 2025: 154646000 # Projected
}

def get_cap_pct(salary, year):
    """Normalizes salary based on that year's cap."""
    cap = salary_cap_history.get(year, 140588000)
    return (salary / cap) * 100

# Example Dataframe Structure
data = {
    'player': ['LeBron James', 'Kevin Durant', 'Step Curry'],
    'year': [2012, 2016, 2024],
    'salary': [17545000, 26540100, 51915615]
}

df = pd.DataFrame(data)

# Applying our Normalizer via Vectorization
df['cap_pct'] = df.apply(lambda row: get_cap_pct(row['salary'], row['year']), axis=1)

print(df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 3.5 MB/s eta 0:00:00
         player  year    salary    cap_pct
0  LeBron James  2012  17545000  30.227069
1  Kevin Durant  2016  26540100  28.191262
2    Step Curry  2024  51915615  36.927487


In [3]:
# 1. Map API Uppercase to Model Lowercase
api_mapping = {
    'PLAYER_AGE': 'age',
    'PTS': 'pts',
    'AST': 'ast',
    'REB': 'trb',
    'USG_PCT': 'usg_pct',
    'TS_PCT': 'ts_pct'
}

df = df.rename(columns=api_mapping)

# 2. Check for missing Advanced Stats (BPM and PER)
# Note: If you don't have these from an 'Advanced' API call, we create
# a simple composite score so the Random Forest still has a column to read.
if 'bpm' not in df.columns:
    # A very rough approximation of impact for training stability
    df['bpm'] = (df['pts'] + df['trb'] + df['ast']) / df['age']

if 'per' not in df.columns:
    # A simple efficiency proxy
    df['per'] = (df['pts'] * df['ts_pct']) + (df['ast'] * 0.5)

# 3. Final sanitization: Keep only what we need and drop NaNs
features = ['age', 'pts', 'ast', 'trb', 'bpm', 'usg_pct', 'ts_pct', 'per']

# Ensure all features exist; if not, initialize with 0
for f in features:
    if f not in df.columns:
        df[f] = 0

# Now your X selection will work
X = df[features]
y = df['cap_pct']

KeyError: 'pts'

In [ ]:
def simulate_team_performance(team_stats, rosters, continuity_dict):
    """
    Simulates win/loss records based on Net Rating and Chemistry.
    """
    results = []

    for team in team_stats:
        # 1. Get Base Net Rating (Offense - Defense)
        base_net_rating = team['ortg'] - team['drtg']

        # 2. Apply Chemistry Multiplier
        # Continuity is a float between 0 and 1
        continuity = continuity_dict.get(team['id'], 0.5)
        chemistry_boost = (continuity - 0.5) * 2.0 # Scale to a small impact

        adj_net_rating = base_net_rating + chemistry_boost

        # 3. Convert Net Rating to Pythagorean Win %
        # We use 14 as our exponent for stability
        exp_win_pct = 0.5 + (0.03 * adj_net_rating) # Linear approximation of Pyth

        # 4. Predict Record at Game 50
        predicted_wins = round(50 * exp_win_pct)
        results.append({
            'team_id': team['id'],
            'wins': predicted_wins,
            'losses': 50 - predicted_wins,
            'net_rating': adj_net_rating
        })

    return pd.DataFrame(results)

In [ ]:
def calculate_desperation(predicted_wins, luxury_tax_bill, gm_tenure):
    """
    Quantifies how likely a GM is to make a "risky" trade.
    """
    # 1. Competitive Gap: Are we underperforming our payroll?
    # Assume 45 wins is the "Safe Zone" for big spenders
    win_gap = max(0, 45 - predicted_wins)

    # 2. Financial Pressure: $1 in tax acts as a multiplier
    tax_pressure = np.log1p(luxury_tax_bill)

    # 3. Job Security: New GMs (tenure < 2) or "Hot Seat" GMs (tenure > 6 without a ring)
    # are more likely to make aggressive moves.
    tenure_multiplier = 1.5 if gm_tenure < 2 else 1.0

    desperation_score = (win_gap * tax_pressure) * tenure_multiplier
    return desperation_score

In [ ]:
def is_cba_compliant(team_a_out, team_b_out, team_a_status, team_b_status):
    """
    Validates if a trade is legal under 2026 CBA rules.
    team_status: 'Below', 'First_Apron', 'Second_Apron'
    """
    # Rule 1: Second Apron aggregation check
    if team_a_status == 'Second_Apron' and len(team_a_out) > 1:
        return False, "Second Apron teams cannot aggregate salaries."

    val_a = sum([p['salary'] for p in team_a_out])
    val_b = sum([p['salary'] for p in team_b_out])

    # Rule 2: Salary Matching Percentages
    # Team A receiving Team B's outgoing salary
    if team_a_status == 'Second_Apron':
        if val_b > val_a: return False, "Second Apron: Cannot take back more salary."
    elif team_a_status == 'First_Apron':
        if val_b > (val_a * 1.10): return False, "First Apron: 110% limit exceeded."
    else: # Below Apron
        if val_b > (val_a * 1.25 + 250000): return False, "Under Apron: 125% limit exceeded."

    return True, "Trade is legal."

In [ ]:
def evaluate_trade_impact(trade_output, team_stats_df):
    """
    Calculates the 'Value Shift' for both teams post-trade.
    """
    impact_report = {}

    for team_id in [trade_output['team_a'], trade_output['team_b']]:
        # 1. Calculate New Team Alpha (Sum of player alphas)
        new_total_alpha = calculate_roster_alpha(trade_output['new_roster'][team_id])

        # 2. Re-run Win Projection
        new_win_projection = simulate_wins(trade_output['new_roster'][team_id])

        # 3. Calculate 'Delta'
        impact_report[team_id] = {
            'alpha_gain': new_total_alpha - trade_output['old_alpha'][team_id],
            'win_gain': new_win_projection - trade_output['old_wins'][team_id]
        }

    return impact_report

In [ ]:
def run_trade_deadline_simulation(player_data, team_metadata, current_year=2026):
    print(f"--- Starting NBA Trade Deadline Engine ({current_year}) ---")

    # STEP 1: Data Normalization (Module 1)
    # Convert nominal salaries to 'Real' Cap Percentages
    player_data['cap_pct'] = player_data.apply(
        lambda x: get_cap_pct(x['salary'], current_year), axis=1
    )

    # STEP 2: The Oracle Valuation (Module 2)
    # Predict 'Fair Value' and calculate Alpha
    player_data['predicted_val'] = oracle.predict(player_data[features])
    player_data['alpha'] = player_data['predicted_val'] - player_data['cap_pct']

    # STEP 3: Season Simulation (Module 3)
    # Identify who is actually winning vs. who is lucky
    standings = simulate_team_performance(team_stats, rosters, continuity_dict)

    # STEP 4: Strategy Assignment (Module 4)
    # Categorize teams as BUYERS or SELLERS based on wins and desperation
    team_strategies = {}
    for _, team in standings.iterrows():
        desperation = calculate_desperation(team['wins'], team['tax'], team['gm_tenure'])
        if team['wins'] > 35:
            team_strategies[team['team_id']] = {'mode': 'BUYER', 'risk': desperation}
        else:
            team_strategies[team['team_id']] = {'mode': 'SELLER', 'risk': desperation}

    # STEP 5: Matching Engine (Module 5)
    # Execute the search for legal, mutually beneficial trades
    proposed_trades = []
    buyers = [t for t, s in team_strategies.items() if s['mode'] == 'BUYER']
    sellers = [t for t, s in team_strategies.items() if s['mode'] == 'SELLER']

    for b_id in buyers:
        for s_id in sellers:
            # Logic: Try to swap high-salary/low-alpha vets for picks/prospects
            trade = attempt_negotiation(b_id, s_id, player_data)
            if trade['legal']:
                proposed_trades.append(trade)

    return proposed_trades

# EXECUTE
final_results = run_trade_deadline_simulation(master_df, team_df)
print(f"Simulation Complete. Found {len(final_results)} viable trades.")

In [ ]:
from google.colab import data_table
import pandas as pd

def visualize_trade_results(proposed_trades):
    """
    Converts the complex trade dictionary into a searchable dashboard.
    """
    trade_list = []

    for trade in proposed_trades:
        # Extracting key data for a summary view
        summary = {
            "Status": "✅ LEGAL" if trade['legal'] else "❌ ILLEGAL",
            "Team A": trade['team_a_name'],
            "Team B": trade['team_b_name'],
            "Outgoing A": ", ".join([p['name'] for p in trade['team_a_out']]),
            "Outgoing B": ", ".join([p['name'] for p in trade['team_b_out']]),
            "Team A Win Gain": f"+{trade['impact']['team_a']['win_gain']:.1f}",
            "Team B Alpha Gain": f"+{trade['impact']['team_b']['alpha_gain']:.2f}",
            "Reason": trade['persona_logic']
        }
        trade_list.append(summary)

    # 1. Create Dataframe
    df_viz = pd.DataFrame(trade_list)

    # 2. Enable Colab's interactive formatter
    data_table.enable_dataframe_formatter()

    # 3. Display (This will create a searchable, sortable table)
    return df_viz

# RUN VISUALIZER
display(visualize_trade_results(final_results))